# FlashVector-GPU — Colab Quickstart

Build the CUDA extension from source and run a zero-copy search against a
PyTorch CUDA tensor. The multi-arch fatbinary covers **sm_75 (T4)**, **sm_80 (A100)**,
**sm_86 (RTX 30xx)**, **sm_89 (L4/RTX 40xx)**, and **sm_90 (H100)**, so no GPU
selection is required.

Runtime → Change runtime type → **GPU** before running the cells below.

In [ ]:
# Cell 1: Install the Rust toolchain and compile the PyO3 extension
!pip install -q maturin
!git clone https://github.com/Sadik00789/FlashVector-GPU.git
%cd FlashVector-GPU
!maturin develop --release

## Verify the assigned GPU
FlashVector-GPU dispatches the matching SM image from the fatbinary automatically.

In [ ]:
import torch

assert torch.cuda.is_available(), "Enable a GPU runtime before continuing"
print(f"Device:     {torch.cuda.get_device_name(0)}")
print(f"Capability: sm_{''.join(str(v) for v in torch.cuda.get_device_capability(0))}")
print(f"Torch:      {torch.__version__}")

## Zero-copy CUDA tensor build + search
10,000 × 128-D vectors are built directly from the device pointer. Queries are
dispatched with `batch_search_device_ptr`, so no host round-trip occurs on the
search path.

In [ ]:
import numpy as np
import torch
import gpu_vector_index

dim = 128
num_vectors = 10_000
top_k = 10

data = torch.randn(num_vectors, dim, device="cuda", dtype=torch.float32)
query = torch.randn(5, dim, device="cuda", dtype=torch.float32)

index = gpu_vector_index.FlashVectorGPU(
    dim=dim, max_elements=num_vectors, m=32, ef_construction=200
)
index.build(data)

labels, dists = index.search(query, top_k=top_k, ef_search=64)
print(f"labels: {labels.shape} {labels.dtype}")
print(f"dists:  {dists.shape} {dists.dtype}")
print(f"Top-5 IDs (query 0): {labels[0][:5]}")
print(f"Top-5 distances:     {dists[0][:5]}")

## Recall sanity check against exact CPU k-NN

In [ ]:
host_data = data.cpu().numpy()
host_query = query[0].cpu().numpy()

exact = np.argsort(((host_data - host_query) ** 2).sum(axis=1))[:top_k]
approx = np.asarray(labels[0])
recall = len(set(exact.tolist()) & set(approx.tolist())) / top_k
print(f"Recall@{top_k}: {recall:.2f}")
assert recall >= 0.9, "Recall below the 0.90 target"

## IVF-PQ ADC path

In [ ]:
ivf_labels, ivf_dists = index.search_ivf(query, top_k=top_k, nprobe=8)
print(f"IVF labels: {ivf_labels.shape}")
print(f"IVF top-5 IDs (query 0): {ivf_labels[0][:5]}")
assert ivf_labels.shape == (5, top_k)
assert np.all(ivf_dists[0][:-1] <= ivf_dists[0][1:]), "distances must be sorted"